In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DoubleType

spark = SparkSession.builder \
    .appName("DE2-Lab0-Validation") \
    .master("local[*]") \
    .getOrCreate()

print("Spark version:", spark.version)
print("Spark UI:", spark.sparkContext.uiWebUrl)

Spark version: 4.0.1
Spark UI: http://192.168.56.1:4040


In [2]:
schema = StructType([
    StructField("id", IntegerType(), False),
    StructField("category", StringType(), True),
    StructField("value", DoubleType(), True),
    StructField("text", StringType(), True),
])

df = spark.read.csv("data/sample.csv", header=True, schema=schema)
print(f"Rows: {df.count()}, Columns: {len(df.columns)}")
df.printSchema()
df.show(5)

Rows: 15, Columns: 4
root
 |-- id: integer (nullable = true)
 |-- category: string (nullable = true)
 |-- value: double (nullable = true)
 |-- text: string (nullable = true)

+---+--------+-----+--------------------+
| id|category|value|                text|
+---+--------+-----+--------------------+
|  1|    tech| 42.5|distributed syste...|
|  2| science| 88.3|machine learning ...|
|  3|    tech| 15.7|spark shuffle ope...|
|  4|business| 67.2|data warehouses s...|
|  5| science| 93.1|clustering algori...|
+---+--------+-----+--------------------+
only showing top 5 rows


In [3]:
df.write.mode("overwrite") \
    .partitionBy("category") \
    .parquet("outputs/lab0/sample_parquet")

print("Parquet written to outputs/lab0/sample_parquet/")

Parquet written to outputs/lab0/sample_parquet/


In [4]:
df_parquet = spark.read.parquet("outputs/lab0/sample_parquet")

print("=== CSV scan plan ===")
df.explain("formatted")

print("\n=== Parquet scan plan ===")
df_parquet.explain("formatted")

=== CSV scan plan ===
== Physical Plan ==
Scan csv  (1)


(1) Scan csv 
Output [4]: [id#0, category#1, value#2, text#3]
Batched: false
Location: InMemoryFileIndex [file:/c:/Users/Enzor/OneDrive/Bureau/ESIEE/E4_Data_Eng_2/Lab0/data/sample.csv]
ReadSchema: struct<id:int,category:string,value:double,text:string>



=== Parquet scan plan ===
== Physical Plan ==
* ColumnarToRow (2)
+- Scan parquet  (1)


(1) Scan parquet 
Output [4]: [id#35, value#36, text#37, category#38]
Batched: true
Location: InMemoryFileIndex [file:/c:/Users/Enzor/OneDrive/Bureau/ESIEE/E4_Data_Eng_2/Lab0/outputs/lab0/sample_parquet]
ReadSchema: struct<id:int,value:double,text:string>

(2) ColumnarToRow [codegen id : 1]
Input [4]: [id#35, value#36, text#37, category#38]




In [5]:
from pyspark.sql import functions as F

agg_df = df_parquet.groupBy("category").agg(
    F.count("*").alias("cnt"),
    F.avg("value").alias("avg_value")
)
agg_df.explain("formatted")
agg_df.show()

== Physical Plan ==
AdaptiveSparkPlan (5)
+- HashAggregate (4)
   +- Exchange (3)
      +- HashAggregate (2)
         +- Scan parquet  (1)


(1) Scan parquet 
Output [2]: [value#36, category#38]
Batched: true
Location: InMemoryFileIndex [file:/c:/Users/Enzor/OneDrive/Bureau/ESIEE/E4_Data_Eng_2/Lab0/outputs/lab0/sample_parquet]
ReadSchema: struct<value:double>

(2) HashAggregate
Input [2]: [value#36, category#38]
Keys [1]: [category#38]
Functions [2]: [partial_count(1), partial_avg(value#36)]
Aggregate Attributes [3]: [count#48L, sum#49, count#50L]
Results [4]: [category#38, count#51L, sum#52, count#53L]

(3) Exchange
Input [4]: [category#38, count#51L, sum#52, count#53L]
Arguments: hashpartitioning(category#38, 200), ENSURE_REQUIREMENTS, [plan_id=88]

(4) HashAggregate
Input [4]: [category#38, count#51L, sum#52, count#53L]
Keys [1]: [category#38]
Functions [2]: [count(1), avg(value#36)]
Aggregate Attributes [2]: [count(1)#45L, avg(value#36)#46]
Results [3]: [category#38, count(1)#45L A

In [6]:
spark.stop()
print("Lab 0 complete. Environment validated.")

Lab 0 complete. Environment validated.
